# scorpus.pairing walkthrough: how pairs are described and sampled

This notebook walks through `scorpus.pairing` on a tiny metadata table (27 cells) so every
result can be checked by eye. Run it top to bottom. With a corpus, the same code runs on
`corpus.metadata_index.df`.

**The whole idea in three steps**

1. **Groups.** Choose the metadata columns that define a group of cells
   (for example "K562, KLF1, dose 10, 24 h").
2. **Pair table.** A list of allowed `source group -> target group`. You start from
   *all* combinations and keep only the rows you want. **Every constraint is "keep the rows
   where this is true".**
3. **Sampler.** Draws actual cells from the allowed pairs (how many cells per side, which
   pairs go together, how cells are linked).

You never need joins or SQL for the common cases. You need about six small pieces of syntax,
all introduced in section 3.

In [1]:
from collections import Counter

import pandas as pd
import polars as pl

from scorpus.pairing import Sampler, combine, cross, describe, links_from, make_groups, src, tgt

## 1. A toy metadata table

Nine kinds of cells, three cells each. Controls have no dose (missing value).

In [2]:
kinds = [
    # cell_line, perturbation, dose, time (h)
    ("K562", "ctrl",       None, 0),
    ("K562", "ctrl",       None, 24),
    ("K562", "KLF1",       1,    24),
    ("K562", "KLF1",       10,   24),
    ("K562", "GATA1",      10,   24),
    ("K562", "KLF1+GATA1", 10,   24),
    ("RPE1", "ctrl",       None, 0),
    ("RPE1", "KLF1",       10,   24),
    ("RPE1", "KLF1",       10,   48),
]
cells = pd.DataFrame([kind for kind in kinds for _ in range(3)],
                     columns=["cell_line", "perturbation", "dose", "time"])
cells["dose"] = cells["dose"].astype("Int64")
cells.insert(0, "cell_id", [f"c{i}" for i in range(len(cells))])
cells.head(8)

,cell_id,cell_line,perturbation,dose,time
0,c0,K562,ctrl,<NA>,0
1,c1,K562,ctrl,<NA>,0
2,c2,K562,ctrl,<NA>,0
3,c3,K562,ctrl,<NA>,24
4,c4,K562,ctrl,<NA>,24
5,c5,K562,ctrl,<NA>,24
6,c6,K562,KLF1,1,24
7,c7,K562,KLF1,1,24


## 2. Step 1: groups

`make_groups(table, by=[...])` puts cells with the same values in the `by` columns into one
group. The result has a small **group table**: one row per group, with an id (`group`), the
`by` columns and the number of cells.

Everything after this works on groups, not cells, so it stays fast with millions of cells.
A column **not** in `by` is mixed inside a group, so put every column you want to
constrain on in `by`.

In [3]:
groups = make_groups(cells, by=["cell_line", "perturbation", "dose", "time"])
groups.table

group,cell_line,perturbation,dose,time,n_cells
i64,str,str,i64,i64,i64
0,"""K562""","""ctrl""",null,0,3
1,"""K562""","""ctrl""",null,24,3
2,"""K562""","""KLF1""",1,24,3
3,"""K562""","""KLF1""",10,24,3
4,"""K562""","""GATA1""",10,24,3
5,"""K562""","""KLF1+GATA1""",10,24,3
6,"""RPE1""","""ctrl""",null,0,3
7,"""RPE1""","""KLF1""",10,24,3
8,"""RPE1""","""KLF1""",10,48,3


`groups.rows[g]` holds the cells (row positions in `cells`) of group `g`:

In [4]:
groups.rows[3], cells.iloc[groups.rows[3]]

(array([ 9, 10, 11]),
    cell_id cell_line perturbation  dose  time
 9       c9      K562         KLF1    10    24
 10     c10      K562         KLF1    10    24
 11     c11      K562         KLF1    10    24)

Two small printing helpers for the rest of the notebook (plain Python, not part of the package):

- `show(pairs)` prints a pair table as readable `source -> target` lines.
- `show_sample(sample)` prints sampled cells with their cell ids and group.

In [5]:
group_of_cell = {int(row): g for g, rows in enumerate(groups.rows) for row in rows}


def label(g):
    row = groups.table.row(g, named=True)  # row g of the group table is group g
    dose = "-" if row["dose"] is None else row["dose"]
    return f"[{g}] {row['cell_line']} {row['perturbation']} dose={dose} t={row['time']}"


def show(pairs, extra=()):
    print(f"{pairs.height} pairs")
    for row in pairs.iter_rows(named=True):
        tags = "  ".join(f"{c}={row[c]}" for c in ["rule", *extra] if c in row)
        print(f"  {label(row['source_group']):<34} -> {label(row['target_group']):<34} {tags}")


def show_sample(sample):
    for source_rows, target_rows in sample:
        source_ids = [cells.cell_id[i] for i in source_rows]
        target_ids = [cells.cell_id[i] for i in target_rows]
        print(f"  {source_ids} {label(group_of_cell[source_rows[0]])}  ->  "
              f"{target_ids} {label(group_of_cell[target_rows[0]])}")

## 3. Step 2: the pair table

### 3.1 Start from everything

`cross(groups)` with no options gives **every** source group combined with **every** target
group: 9 x 9 = 81 rows. Each row repeats both groups' columns, with the prefix `source_` or
`target_`.

In [6]:
everything = cross(groups)
print(everything.height, "rows")
everything.head(3)

81 rows


source_group,source_cell_line,source_perturbation,source_dose,source_time,source_n_cells,target_group,target_cell_line,target_perturbation,target_dose,target_time,target_n_cells
i64,str,str,i64,i64,i64,i64,str,str,i64,i64,i64
0,"""K562""","""ctrl""",null,0,3,0,"""K562""","""ctrl""",null,0,3
0,"""K562""","""ctrl""",null,0,3,1,"""K562""","""ctrl""",null,24,3
0,"""K562""","""ctrl""",null,0,3,2,"""K562""","""KLF1""",1,24,3


Every constraint below just removes rows from this table.

### 3.2 The only syntax you need

| You write | It means |
|---|---|
| `pl.col("perturbation")` | the column `perturbation` |
| `pl.col("perturbation") == "ctrl"` | yes/no for each row: is it a control? Also `!=`, `>`, `>=`, `<`, `<=` |
| `a & b`, `a \| b`, `~a` | and, or, not. **Put each comparison in brackets**: `(pl.col("time") > 0) & (pl.col("dose") == 10)` |
| `pl.col("perturbation").is_in(["KLF1", "GATA1"])` | is the value one of these? |
| `pl.col("dose").is_null()` | is the value missing? |
| `src.time`, `tgt.time` | short for `pl.col("source_time")`, `pl.col("target_time")` in a pair table |

A condition is just a value you can store in a variable and reuse. `.filter(condition)` keeps
the rows where it is true:

In [7]:
is_ctrl = pl.col("perturbation") == "ctrl"
groups.table.filter(is_ctrl)

group,cell_line,perturbation,dose,time,n_cells
i64,str,str,i64,i64,i64
0,"""K562""","""ctrl""",null,0,3
1,"""K562""","""ctrl""",null,24,3
6,"""RPE1""","""ctrl""",null,0,3


### 3.3 Constraint type A: who can be a source, who can be a target

`cross(groups, source=..., target=...)`: conditions on a group by itself, checked before
combining.

*Controls are sources, perturbed groups are targets:*

In [8]:
show(cross(groups, source=is_ctrl, target=~is_ctrl))

18 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [0] K562 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [1] K562 ctrl dose=- t=24          -> [7] RPE1 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [8] RPE1 KLF1 dose=10 t=48         
  [6] RPE1 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [

That gives 3 control groups x 6 perturbed groups = 18 pairs. Note that it pairs RPE1 controls
with K562 cells. We fix that next.

Conditions combine freely. *Only single perturbations at 24 h as targets:*

In [9]:
single_24h = ~is_ctrl & ~pl.col("perturbation").str.contains("+", literal=True) & (pl.col("time") == 24)
show(cross(groups, source=is_ctrl, target=single_24h))

12 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [0] K562 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [7] RPE1 KLF1 dose=10 t=24         
  [6] RPE1 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [6] RPE1 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [6] RPE1 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         


### 3.4 Constraint type B: values that must be the same on both sides

`on=[...]`: the listed columns must be equal between source and target. This is the most
common constraint ("same cell line", "same donor", "same batch").

*Controls -> perturbed, same cell line:*

In [10]:
basic = cross(groups, on=["cell_line"], source=is_ctrl, target=~is_ctrl)
show(basic)

10 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         


Several columns: *same cell line and same time point*. Now each perturbed group only gets the
control measured at its own time, and the RPE1 targets get nothing (no RPE1 control at
24 h or 48 h):

In [11]:
same_time = cross(groups, on=["cell_line", "time"], source=is_ctrl, target=~is_ctrl)
show(same_time)

4 pairs
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   


### 3.5 Constraint type C: compare the two sides

`.filter(...)` after `cross`, using `src.<column>` and `tgt.<column>`. Use this for anything
other than "equal": later in time, higher dose, different perturbation.

*Same cell line, perturbation and dose, target later in time* (a time course):

In [12]:
time_forward = cross(groups, on=["cell_line", "perturbation", "dose"]).filter(tgt.time > src.time)
show(time_forward)

2 pairs
  [0] K562 ctrl dose=- t=0           -> [1] K562 ctrl dose=- t=24          
  [7] RPE1 KLF1 dose=10 t=24         -> [8] RPE1 KLF1 dose=10 t=48         


*Same cell line, perturbation and time, target at a higher dose* (dose response):

In [13]:
dose_up = cross(groups, on=["cell_line", "perturbation", "time"], source=~is_ctrl).filter(tgt.dose > src.dose)
show(dose_up)

1 pairs
  [2] K562 KLF1 dose=1 t=24          -> [3] K562 KLF1 dose=10 t=24         


*Perturbed -> a different perturbation in the same cell line, time and dose*
(perturbation-to-perturbation transfer):

In [14]:
show(cross(groups, on=["cell_line", "time", "dose"], source=~is_ctrl, target=~is_ctrl)
     .filter(tgt.perturbation != src.perturbation))

6 pairs
  [3] K562 KLF1 dose=10 t=24         -> [4] K562 GATA1 dose=10 t=24        
  [3] K562 KLF1 dose=10 t=24         -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [4] K562 GATA1 dose=10 t=24        -> [3] K562 KLF1 dose=10 t=24         
  [4] K562 GATA1 dose=10 t=24        -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [5] K562 KLF1+GATA1 dose=10 t=24   -> [3] K562 KLF1 dose=10 t=24         
  [5] K562 KLF1+GATA1 dose=10 t=24   -> [4] K562 GATA1 dose=10 t=24        


**Watch out for missing values.** In `.filter`, `src.dose == tgt.dose` is *not true* when both
doses are missing, so control -> control pairs disappear. `on=` treats missing = missing
as equal. In a filter, use `.eq_missing` to get the same behaviour:

In [15]:
print("filter with == :", cross(groups, source=is_ctrl, target=is_ctrl).filter(src.dose == tgt.dose).height, "pairs")
print("filter with eq_missing :", cross(groups, source=is_ctrl, target=is_ctrl).filter(src.dose.eq_missing(tgt.dose)).height, "pairs")
print("on=['dose'] :", cross(groups, on=["dose"], source=is_ctrl, target=is_ctrl).height, "pairs")

filter with == : 0 pairs
filter with eq_missing : 9 pairs
on=['dose'] : 9 pairs


### 3.6 Constraint type D: remove particular pairs

Put `~` (not) in front of a condition describing what you do *not* want.

*Basic pairs, but no RPE1 48 h targets:*

In [16]:
show(basic.filter(~((tgt.cell_line == "RPE1") & (tgt.time == 48))))

9 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         


Or by group id when you just want to drop specific rows you saw in `show`:

In [17]:
show(basic.filter(~pl.col("target_group").is_in([5, 8])))

7 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         


### 3.7 Several rules at once: `combine`

Build one table per rule, give each a label, and stack them with `combine`. Adding the label is
`table.with_columns(rule=pl.lit("name"))` (`pl.lit` means "this fixed value in every row").

*A development data set: controls at t=0 -> anything later (same cell line), plus the time
course and the dose response from above.*

In [18]:
from_origin = cross(groups, on=["cell_line"], source=is_ctrl & (pl.col("time") == 0), target=pl.col("time") > 0)
development = combine(
    from_origin.with_columns(rule=pl.lit("from_origin")),
    time_forward.with_columns(rule=pl.lit("time_course")),
    dose_up.with_columns(rule=pl.lit("dose_response")),
)
show(development)

9 pairs
  [0] K562 ctrl dose=- t=0           -> [1] K562 ctrl dose=- t=24          rule=from_origin
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          rule=from_origin
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         rule=from_origin
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        rule=from_origin
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   rule=from_origin
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         rule=from_origin
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         rule=from_origin
  [7] RPE1 KLF1 dose=10 t=24         -> [8] RPE1 KLF1 dose=10 t=48         rule=time_course
  [2] K562 KLF1 dose=1 t=24          -> [3] K562 KLF1 dose=10 t=24         rule=dose_response


**If the same pair is allowed by two rules, the first rule's row is kept.** Here K562 ctrl t0
-> K562 ctrl t24 is allowed by both `from_origin` and `time_course`; it is labelled
`from_origin` because that rule comes first. Swap the order and the label changes:

In [19]:
swapped = combine(time_forward.with_columns(rule=pl.lit("time_course")),
                  from_origin.with_columns(rule=pl.lit("from_origin")))
show(swapped.filter((src.perturbation == "ctrl") & (tgt.perturbation == "ctrl")))

1 pairs
  [0] K562 ctrl dose=- t=0           -> [1] K562 ctrl dose=- t=24          rule=time_course


`describe` summarises a pair table per rule and tells you how many groups are never used:

In [20]:
print(describe(groups, development))

cells: 27   groups: 9   pairs: 9
rule                             pairs   source groups   target groups
from_origin                          7               2               7
time_course                          1               1               1
dose_response                        1               1               1
groups never a source: 5   groups never a target: 2


### 3.8 "Use the best partner, otherwise a fallback"

*Pair each perturbed group with the control from the same time point. If there is none, use
any control of the same cell line.*

Build the strict table and the loose table, then drop from the loose table every target that
already has a partner in the strict one:

In [21]:
strict = cross(groups, on=["cell_line", "time"], source=is_ctrl, target=~is_ctrl)
loose = cross(groups, on=["cell_line"], source=is_ctrl, target=~is_ctrl)
fallback = loose.filter(~pl.col("target_group").is_in(strict["target_group"].to_list()))

show(combine(strict.with_columns(rule=pl.lit("same_time")),
             fallback.with_columns(rule=pl.lit("fallback"))))

6 pairs
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          rule=same_time
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         rule=same_time
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        rule=same_time
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   rule=same_time
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         rule=fallback
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         rule=fallback


K562 targets only get the 24 h control; the RPE1 targets, which have no control at their time,
fall back to the 0 h control. (Compare with `basic`, where K562 targets also got the 0 h
control.)

### 3.9 Constraints that need a value computed per group first

Some rules depend on the other groups, for example "the next time point" or "the highest
dose". Add a column to the **group table** first with `groups.with_columns(...)`, then use it
like any other column.

`.over([...])` means "compute this separately within each combination of these columns".

*Only the highest dose of each perturbation as target:*

In [22]:
groups = groups.with_columns(top_dose=pl.col("dose").max().over(["cell_line", "perturbation", "time"]))
show(cross(groups, on=["cell_line", "time"], source=is_ctrl, target=~is_ctrl & (pl.col("dose") == pl.col("top_dose"))))

3 pairs
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   


*Only the next time point* (not every later one). `shift(-1)` takes the value from the next
row, `order_by="time"` sorts by time first:

In [23]:
groups = groups.with_columns(
    next_time=pl.col("time").shift(-1).over(["cell_line", "perturbation", "dose"], order_by="time")
)
groups.table

group,cell_line,perturbation,dose,time,n_cells,top_dose,next_time
i64,str,str,i64,i64,i64,i64,i64
0,"""K562""","""ctrl""",null,0,3,null,24
1,"""K562""","""ctrl""",null,24,3,null,null
2,"""K562""","""KLF1""",1,24,3,10,null
3,"""K562""","""KLF1""",10,24,3,10,null
4,"""K562""","""GATA1""",10,24,3,10,null
5,"""K562""","""KLF1+GATA1""",10,24,3,10,null
6,"""RPE1""","""ctrl""",null,0,3,null,null
7,"""RPE1""","""KLF1""",10,24,3,10,48
8,"""RPE1""","""KLF1""",10,48,3,10,null


In [24]:
show(cross(groups, on=["cell_line", "perturbation", "dose"]).filter(tgt.time == src.next_time))

2 pairs
  [0] K562 ctrl dose=- t=0           -> [1] K562 ctrl dose=- t=24          
  [7] RPE1 KLF1 dose=10 t=24         -> [8] RPE1 KLF1 dose=10 t=48         


### 3.10 Combinations: single perturbation -> a combination that contains it

`.str.split("+")` turns `"KLF1+GATA1"` into the list `["KLF1", "GATA1"]`, and
`.list.contains(...)` checks if a value is in that list:

In [25]:
is_combo = pl.col("perturbation").str.contains("+", literal=True)
add_one = cross(groups, on=["cell_line", "time"], source=~is_ctrl & ~is_combo, target=is_combo).filter(
    tgt.perturbation.str.split("+").list.contains(src.perturbation)
)
show(add_one)

3 pairs
  [2] K562 KLF1 dose=1 t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [3] K562 KLF1 dose=10 t=24         -> [5] K562 KLF1+GATA1 dose=10 t=24   
  [4] K562 GATA1 dose=10 t=24        -> [5] K562 KLF1+GATA1 dose=10 t=24   


Add `"dose"` to `on=` to keep only the dose-10 single as source.

### 3.11 Weights: how often each pair is drawn

Any number column on the pair table can be a weight (section 4.6 uses it). `pl.when(...).then(...).otherwise(...)`
is "if / else" per row.

*Draw GATA1 pairs three times as often:*

In [26]:
weighted = basic.with_columns(weight=pl.when(tgt.perturbation == "GATA1").then(3.0).otherwise(1.0))
show(weighted, extra=["weight"])

10 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          weight=1.0
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         weight=1.0
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        weight=3.0
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   weight=1.0
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          weight=1.0
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         weight=1.0
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        weight=3.0
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   weight=1.0
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         weight=1.0
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         weight=1.0


*Give every target group the same total weight* (a target with 2 possible controls gets 1/2
per pair). `pl.len().over("target_group")` counts the pairs of each target group:

In [27]:
balanced = basic.with_columns(weight=1 / pl.len().over("target_group"))
show(balanced, extra=["weight"])

10 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          weight=0.5
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         weight=0.5
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        weight=0.5
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   weight=0.5
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          weight=0.5
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         weight=0.5
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        weight=0.5
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   weight=0.5
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         weight=1.0
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         weight=1.0


### 3.12 Checking what you built

Besides `show` and `describe`, list the perturbed groups that never appear as a target.
These are usually the ones you forgot about. For `same_time` it finds the two RPE1 groups,
which have no control at their time point:

In [28]:
never_target = groups.table.filter(~is_ctrl & ~pl.col("group").is_in(same_time["target_group"].to_list()))
never_target.select("group", "cell_line", "perturbation", "dose", "time")

group,cell_line,perturbation,dose,time
i64,str,str,i64,i64
7,"""RPE1""","""KLF1""",10,24
8,"""RPE1""","""KLF1""",10,48


### 3.13 Cheat sheet

| I want ... | Write |
|---|---|
| sources must be controls | `cross(groups, source=pl.col("perturbation") == "ctrl", ...)` |
| targets must not be controls | `target=pl.col("perturbation") != "ctrl"` |
| same cell line / donor / batch | `on=["cell_line"]` |
| target later in time | `.filter(tgt.time > src.time)` |
| target at a higher dose | `.filter(tgt.dose > src.dose)` |
| different perturbation | `.filter(tgt.perturbation != src.perturbation)` |
| equal, including missing = missing, in a filter | `.filter(src.dose.eq_missing(tgt.dose))` |
| drop some pairs | `.filter(~(condition))` |
| drop some target groups by id | `.filter(~pl.col("target_group").is_in([5, 8]))` |
| several rules | `combine(a.with_columns(rule=pl.lit("a")), b.with_columns(rule=pl.lit("b")))`; first rule wins |
| fallback when the strict rule finds nothing | `loose.filter(~pl.col("target_group").is_in(strict["target_group"].to_list()))` |
| a per-group value (max dose, next time) | `groups = groups.with_columns(name=... .over([...]))`, then use `name` |
| single -> combination containing it | `.filter(tgt.perturbation.str.split("+").list.contains(src.perturbation))` |
| weights | `.with_columns(weight=pl.when(cond).then(3.0).otherwise(1.0))` |

## 4. Step 3: sampling cells

`Sampler(groups, pairs, ...)` turns allowed group pairs into cells. A **sample** is a list of
`(source rows, target rows)` tuples; rows are positions in `cells`. All draws are reproducible
from `(seed, step)`.

### 4.1 One cell per side

In [29]:
sampler = Sampler(groups, basic, n_source=1)
for step in range(3):
    print("step", step, sampler.sample(step))
    show_sample(sampler.sample(step))

step 0 [([4], [12])]
  ['c4'] [1] K562 ctrl dose=- t=24  ->  ['c12'] [4] K562 GATA1 dose=10 t=24
step 1 [([20], [22])]
  ['c20'] [6] RPE1 ctrl dose=- t=0  ->  ['c22'] [7] RPE1 KLF1 dose=10 t=24
step 2 [([0], [7])]
  ['c0'] [0] K562 ctrl dose=- t=0  ->  ['c7'] [2] K562 KLF1 dose=1 t=24


### 4.2 Sets of cells: `n_source`, `n_target`

`n_source=2`: 2 source and 2 target cells. `n_target` sets the target side separately, and
`"all"` takes every cell of the group.

In [30]:
print("n_source=2:")
show_sample(Sampler(groups, basic, n_source=2).sample(0))
print("n_source=2, n_target=3:")
show_sample(Sampler(groups, basic, n_source=2, n_target=3).sample(0))
print('n_source="all", n_target=1:')
show_sample(Sampler(groups, basic, n_source="all", n_target=1).sample(0))

n_source=2:
  ['c3', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c14', 'c12'] [4] K562 GATA1 dose=10 t=24
n_source=2, n_target=3:
  ['c3', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c12', 'c14', 'c13'] [4] K562 GATA1 dose=10 t=24
n_source="all", n_target=1:
  ['c5', 'c3', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c12'] [4] K562 GATA1 dose=10 t=24


If a group has fewer cells than asked for, it gives all the cells it has (here 3 instead of 4),
and padding is left to the model. `drop_small=True` leaves out pair rows whose groups are too
small instead; in this toy table every group has 3 cells, so nothing is left:

In [31]:
show_sample(Sampler(groups, basic, n_source=4).sample(0))
print("pair rows left with drop_small:", Sampler(groups, basic, n_source=4, drop_small=True).n_bundles)

  ['c5', 'c3', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c14', 'c13', 'c12'] [4] K562 GATA1 dose=10 t=24
pair rows left with drop_small: 0


### 4.3 Several pairs in one sample: `bundle_by`

`bundle_by=[pair-table columns]`: pair rows with the same values in these columns are drawn
together, as one sample. A group used by several of these rows gives **the same cells** to
all of them.

*All doses of one perturbation together, sharing the same control cells:* in `same_time`,
the K562 ctrl 24 h group pairs with KLF1 dose 1 and KLF1 dose 10.

In [32]:
dose_series = Sampler(groups, same_time, n_source=2, bundle_by=["source_group", "target_perturbation"])
print(dose_series.n_bundles, "bundles")
for step in range(4):
    print("step", step)
    show_sample(dose_series.sample(step))

3 bundles
step 0
  ['c3', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c14', 'c12'] [4] K562 GATA1 dose=10 t=24
step 1
  ['c5', 'c4'] [1] K562 ctrl dose=- t=24  ->  ['c16', 'c15'] [5] K562 KLF1+GATA1 dose=10 t=24
step 2
  ['c4', 'c3'] [1] K562 ctrl dose=- t=24  ->  ['c8', 'c7'] [2] K562 KLF1 dose=1 t=24
  ['c4', 'c3'] [1] K562 ctrl dose=- t=24  ->  ['c11', 'c9'] [3] K562 KLF1 dose=10 t=24
step 3
  ['c4', 'c5'] [1] K562 ctrl dose=- t=24  ->  ['c16', 'c15'] [5] K562 KLF1+GATA1 dose=10 t=24


The KLF1 bundle has two pair rows (dose 1 and dose 10); whenever it is drawn, both rows get
the same two control cells. The other bundles have one row each.

### 4.4 Cell-to-cell links: `links`

- `links="none"` (default): the sample is a set of source cells and a set of target cells.
- `links="random"`: each source cell gets its own random partner from the target group, so
  the sample is a list of single-cell pairs.

In [33]:
show_sample(Sampler(groups, basic, n_source=3, links="random").sample(0))

  ['c5'] [1] K562 ctrl dose=- t=24  ->  ['c12'] [4] K562 GATA1 dose=10 t=24
  ['c3'] [1] K562 ctrl dose=- t=24  ->  ['c12'] [4] K562 GATA1 dose=10 t=24
  ['c4'] [1] K562 ctrl dose=- t=24  ->  ['c12'] [4] K562 GATA1 dose=10 t=24


Partners are drawn independently for each source cell, so two source cells can get the same
partner (here the target group only has three cells).

- A **link table** with columns `source_row` and `target_row` gives real partners, for
  example lineage (parent -> child) or links computed by optimal transport. Cells without a
  link to the partner group are skipped.

*Pretend lineage: the K562 0 h controls c0, c1, c2 are the parents of c3, c4, c5.*

In [34]:
lineage = pd.DataFrame({"source_row": [0, 1, 2], "target_row": [3, 4, 5]})
ctrl_time_course = time_forward.filter(src.perturbation == "ctrl")
show(ctrl_time_course)
show_sample(Sampler(groups, ctrl_time_course, n_source=3, links=lineage).sample(0))

1 pairs
  [0] K562 ctrl dose=- t=0           -> [1] K562 ctrl dose=- t=24          
  ['c2'] [0] K562 ctrl dose=- t=0  ->  ['c5'] [1] K562 ctrl dose=- t=24
  ['c0'] [0] K562 ctrl dose=- t=0  ->  ['c3'] [1] K562 ctrl dose=- t=24
  ['c1'] [0] K562 ctrl dose=- t=0  ->  ['c4'] [1] K562 ctrl dose=- t=24


- `links="self"`: each cell is its own partner (source and target group must be the same).

**Different links for different pair rows.** Put the link kind in a pair-table column and pass
the column name. This is pertTF's default training pairing: controls go to a random cell of a
perturbation in the same cell line, and perturbed cells are paired with themselves.

In [35]:
perttf_pairs = combine(
    cross(groups, on=["cell_line"], source=is_ctrl, target=~is_ctrl).with_columns(link=pl.lit("random")),
    cross(groups, on=["cell_line", "perturbation", "dose", "time"], source=~is_ctrl).with_columns(link=pl.lit("self")),
)
show(perttf_pairs, extra=["link"])
for sample in Sampler(groups, perttf_pairs, links="link").sample_batch(step=0, n_bundles=6):
    show_sample(sample)

16 pairs
  [0] K562 ctrl dose=- t=0           -> [2] K562 KLF1 dose=1 t=24          link=random
  [0] K562 ctrl dose=- t=0           -> [3] K562 KLF1 dose=10 t=24         link=random
  [0] K562 ctrl dose=- t=0           -> [4] K562 GATA1 dose=10 t=24        link=random
  [0] K562 ctrl dose=- t=0           -> [5] K562 KLF1+GATA1 dose=10 t=24   link=random
  [1] K562 ctrl dose=- t=24          -> [2] K562 KLF1 dose=1 t=24          link=random
  [1] K562 ctrl dose=- t=24          -> [3] K562 KLF1 dose=10 t=24         link=random
  [1] K562 ctrl dose=- t=24          -> [4] K562 GATA1 dose=10 t=24        link=random
  [1] K562 ctrl dose=- t=24          -> [5] K562 KLF1+GATA1 dose=10 t=24   link=random
  [6] RPE1 ctrl dose=- t=0           -> [7] RPE1 KLF1 dose=10 t=24         link=random
  [6] RPE1 ctrl dose=- t=0           -> [8] RPE1 KLF1 dose=10 t=48         link=random
  [2] K562 KLF1 dose=1 t=24          -> [2] K562 KLF1 dose=1 t=24          link=self
  [3] K562 KLF1 dose=10 t=24        

**Optimal transport links with `links_from`.** Build the pair table first, then compute cell links
only inside the allowed pairs with your own matching function. `match(source_rows, target_rows)`
returns positions `(i, j)`: `source_rows[i[k]]` is linked to `target_rows[j[k]]`. Here the cells
get made-up 2-D coordinates (in practice a PCA of expression) and are matched one to one by
distance. The rows that should use these links get `link="table"`:

In [36]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

coords = np.random.default_rng(0).normal(size=(len(cells), 2))
ctrl_to_pert = cross(groups, on=["cell_line"], source=is_ctrl, target=~is_ctrl)
ot_links = links_from(groups, ctrl_to_pert, lambda s, t: linear_sum_assignment(cdist(coords[s], coords[t])))
print(ot_links.height, "links, for example:", ot_links.head(3).rows())

ot_pairs = combine(
    ctrl_to_pert.with_columns(link=pl.lit("table")),
    cross(groups, on=["cell_line", "perturbation", "dose", "time"], source=~is_ctrl).with_columns(link=pl.lit("self")),
)
for step in range(3):
    show_sample(Sampler(groups, ot_pairs, n_source=2, links="link", link_table=ot_links).sample(step))

30 links, for example: [(0, 7), (1, 8), (2, 6)]
  ['c6'] [2] K562 KLF1 dose=1 t=24  ->  ['c6'] [2] K562 KLF1 dose=1 t=24
  ['c7'] [2] K562 KLF1 dose=1 t=24  ->  ['c7'] [2] K562 KLF1 dose=1 t=24
  ['c23'] [7] RPE1 KLF1 dose=10 t=24  ->  ['c23'] [7] RPE1 KLF1 dose=10 t=24
  ['c22'] [7] RPE1 KLF1 dose=10 t=24  ->  ['c22'] [7] RPE1 KLF1 dose=10 t=24
  ['c1'] [0] K562 ctrl dose=- t=0  ->  ['c9'] [3] K562 KLF1 dose=10 t=24
  ['c0'] [0] K562 ctrl dose=- t=0  ->  ['c10'] [3] K562 KLF1 dose=10 t=24


### 4.5 Many samples at once: `sample_batch`

In [37]:
for sample in Sampler(groups, basic).sample_batch(step=0, n_bundles=4):
    show_sample(sample)

  ['c3'] [1] K562 ctrl dose=- t=24  ->  ['c13'] [4] K562 GATA1 dose=10 t=24
  ['c2'] [0] K562 ctrl dose=- t=0  ->  ['c13'] [4] K562 GATA1 dose=10 t=24
  ['c1'] [0] K562 ctrl dose=- t=0  ->  ['c8'] [2] K562 KLF1 dose=1 t=24
  ['c2'] [0] K562 ctrl dose=- t=0  ->  ['c8'] [2] K562 KLF1 dose=1 t=24


### 4.6 Weights: `weight="column"`

Count how often each target group is drawn in 6000 samples, without and with the weights
from section 3.11:

In [38]:
def target_counts(pairs, weight=None):
    batch = Sampler(groups, pairs, weight=weight).sample_batch(step=0, n_bundles=6000)
    counts = Counter(label(group_of_cell[target_rows[0]]) for ((source_rows, target_rows),) in batch)
    return dict(sorted(counts.items()))


print("uniform (each pair row equally likely):")
for name, n in target_counts(basic).items():
    print(f"  {n:5}  {name}")
print("GATA1 x3:")
for name, n in target_counts(weighted, "weight").items():
    print(f"  {n:5}  {name}")
print("each target group equally likely:")
for name, n in target_counts(balanced, "weight").items():
    print(f"  {n:5}  {name}")

uniform (each pair row equally likely):
   1233  [2] K562 KLF1 dose=1 t=24
   1233  [3] K562 KLF1 dose=10 t=24
   1122  [4] K562 GATA1 dose=10 t=24
   1189  [5] K562 KLF1+GATA1 dose=10 t=24
    654  [7] RPE1 KLF1 dose=10 t=24
    569  [8] RPE1 KLF1 dose=10 t=48
GATA1 x3:
    852  [2] K562 KLF1 dose=1 t=24
    890  [3] K562 KLF1 dose=10 t=24
   2522  [4] K562 GATA1 dose=10 t=24
    903  [5] K562 KLF1+GATA1 dose=10 t=24
    427  [7] RPE1 KLF1 dose=10 t=24
    406  [8] RPE1 KLF1 dose=10 t=48
each target group equally likely:
   1040  [2] K562 KLF1 dose=1 t=24
   1017  [3] K562 KLF1 dose=10 t=24
    999  [4] K562 GATA1 dose=10 t=24
    941  [5] K562 KLF1+GATA1 dose=10 t=24
   1016  [7] RPE1 KLF1 dose=10 t=24
    987  [8] RPE1 KLF1 dose=10 t=48


Without weights, K562 targets are drawn twice as often as RPE1 targets because they have two
possible controls (two pair rows each).

### 4.7 Epoch mode: every cell once

`sample` draws pairs at random (with replacement). `epoch(epoch)` instead uses every cell of
every source group exactly once per epoch: each group is shuffled and cut into chunks of
`n_source` cells, and each chunk is paired by drawing one of its allowed target groups.
With 3 cells per group and `n_source=2`, every group gives a chunk of 2 and a chunk of 1
(the last chunk keeps what is left).

In [39]:
epoch_samples = Sampler(groups, basic, n_source=2).epoch(0)
print(len(epoch_samples), "samples")
for sample in epoch_samples[:4]:
    show_sample(sample)

used = Counter(cells.cell_id[i] for sample in epoch_samples for source_rows, _ in sample for i in source_rows)
print("source cells used:", dict(sorted(used.items())))

6 samples
  ['c3', 'c5'] [1] K562 ctrl dose=- t=24  ->  ['c12', 'c14'] [4] K562 GATA1 dose=10 t=24
  ['c1'] [0] K562 ctrl dose=- t=0  ->  ['c7', 'c8'] [2] K562 KLF1 dose=1 t=24
  ['c4'] [1] K562 ctrl dose=- t=24  ->  ['c15', 'c16'] [5] K562 KLF1+GATA1 dose=10 t=24
  ['c19'] [6] RPE1 ctrl dose=- t=0  ->  ['c25', 'c24'] [8] RPE1 KLF1 dose=10 t=48
source cells used: {'c0': 1, 'c1': 1, 'c18': 1, 'c19': 1, 'c2': 1, 'c20': 1, 'c3': 1, 'c4': 1, 'c5': 1}


`cover="target"` makes every target cell appear once instead:

In [40]:
epoch_samples = Sampler(groups, basic, n_source=2).epoch(0, cover="target")
used = Counter(cells.cell_id[i] for sample in epoch_samples for _, target_rows in sample for i in target_rows)
print(len(used), "target cells, each used", set(used.values()), "time(s)")

18 target cells, each used {1} time(s)


`drop_last=True` leaves out each group's short last chunk (here the chunks of 1 cell), so every
sample has exactly `n_source` cells; without it, padding the short chunks is up to the model:

In [41]:
print("chunk sizes:", Counter(len(s) for sample in Sampler(groups, basic, n_source=2).epoch(0) for s, _ in sample))
print("with drop_last:", Counter(len(s) for sample in Sampler(groups, basic, n_source=2).epoch(0, drop_last=True) for s, _ in sample))

chunk sizes: Counter({2: 3, 1: 3})
with drop_last: Counter({2: 3})


### 4.8 Sampler options at a glance

| Option | Meaning |
|---|---|
| `n_source`, `n_target` | cells per source / target group in a sample: a number or `"all"`; `n_target` defaults to `n_source` |
| `bundle_by` | pair-table columns; rows with equal values are drawn together and share cells |
| `links` | `"none"` (sets), `"random"` (one random partner per cell), `"self"` (the same cell), a link table, or a pair-table column giving the kind per row (`"table"` rows use `link_table`) |
| `link_table` | columns `source_row`, `target_row` and optionally `weight` (lineage, OT; see `links_from`, which can keep each cell's `top_k` heaviest links) |
| `drop_small` | leave out pair rows whose groups have fewer cells than `n_source` / `n_target` |
| `weight` | a pair-table column; a bundle's weight is the sum of its rows |
| `sample(step, seed)` | one sample |
| `sample_batch(step, n_bundles, seed)` | many samples at once (fastest for single cells) |
| `epoch(epoch, seed, cover, drop_last)` | every cell of the covered side once; `drop_last` leaves out short last chunks |

## 5. Common mistakes

1. **Forgetting `on=`.** `cross(groups, source=is_ctrl, target=~is_ctrl)` pairs RPE1 controls
   with K562 cells (section 3.3). Check with `show` or `describe`.
2. **Missing values in filters.** `src.dose == tgt.dose` drops missing = missing pairs; use
   `on=` or `.eq_missing` (section 3.5).
3. **Brackets around comparisons.** `pl.col("time") > 0 & is_ctrl` fails or does the wrong
   thing; write `(pl.col("time") > 0) & is_ctrl`.
4. **Rule order in `combine`.** The first rule that allows a pair keeps it (section 3.7).
5. **Groups too coarse.** A column that is not in `make_groups(by=...)` cannot be constrained
   on and is mixed inside each group.